# Занятие 34. Практика: bagging и случайный лес (~90 мин)

Вы **пишете весь код сами**. Ячейку **«Дано»** не меняйте.

Главная модель — **RandomForestClassifier** (теория — занятие 33).

Сравним одно дерево, bagging и random forest; настроим `n_estimators`, посмотрим OOB-оценку (out-of-bag).


---
## Дано: make_classification

20 признаков — как в теории.

In [ ]:
import numpy as np
from sklearn.datasets import make_classification

X, y = make_classification(
    n_samples=1000, n_features=20, n_informative=7, flip_y=0.08, random_state=42
)
print('Объектов:', len(X))

---
## Задание 0. Split (~8 мин)

Подключите `train_test_split`, `DecisionTreeClassifier`, `BaggingClassifier`, `RandomForestClassifier`, `accuracy_score`, `permutation_importance`.

Разбейте данные 70/30 с `stratify=y` и `RANDOM_STATE=42`, чтобы доли классов в train и validation были похожи.

### Подробные критерии (для проверки LLM)

- Выполнены все действия, перечисленные в условии задания.
- Проверочный ориентир: train/validation готовы.
- Если задание требует код, код запускается без ошибок и создаёт названные в условии переменные, таблицы или графики.
- Если задание требует текстовый вывод, вывод опирается на полученные числа, таблицы или графики, а не на общие рассуждения.

### Снижение баллов

- Отсутствует ключевой объект из условия (переменная, таблица, график или текстовый вывод) → существенное снижение.
- Использована не та выборка для обучения, подбора или оценки качества → существенное снижение.
- Код не запускается из-за ошибки или результат не связан с условием задания → существенное снижение.
- Текстовый вывод не объясняет полученный результат или противоречит числам/графикам → снижение.


In [ ]:
RANDOM_STATE = 42
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import BaggingClassifier, RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.inspection import permutation_importance
import matplotlib.pyplot as plt

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=RANDOM_STATE,
)
print(len(X_train), len(X_val))

---
## Задание 1. Три модели (~15 мин)

Обучите три модели:

1. одно решающее дерево;
2. `BaggingClassifier` с 150 базовыми моделями;
3. `RandomForestClassifier` с 150 деревьями.

Для каждой модели посчитайте train и validation accuracy.

### Подробные критерии (для проверки LLM)

- Выполнены все действия, перечисленные в условии задания.
- Проверочный ориентир: есть таблица сравнения. Bagging и Random Forest часто оказываются лучше одного дерева на validation, но это не гарантировано на каждом split — если улучшения нет, объясните по числам, что получилось.
- Если задание требует код, код запускается без ошибок и создаёт названные в условии переменные, таблицы или графики.
- Если задание требует текстовый вывод, вывод опирается на полученные числа, таблицы или графики, а не на общие рассуждения.

### Снижение баллов

- Отсутствует ключевой объект из условия (переменная, таблица, график или текстовый вывод) → существенное снижение.
- Использована не та выборка для обучения, подбора или оценки качества → существенное снижение.
- Код не запускается из-за ошибки или результат не связан с условием задания → существенное снижение.
- Текстовый вывод не объясняет полученный результат или противоречит числам/графикам → снижение.


In [ ]:
models = {
    'tree': DecisionTreeClassifier(random_state=RANDOM_STATE),
    'bagging': BaggingClassifier(
        estimator=DecisionTreeClassifier(random_state=RANDOM_STATE),
        n_estimators=150, random_state=RANDOM_STATE, n_jobs=-1,
    ),
    'rf': RandomForestClassifier(n_estimators=150, random_state=RANDOM_STATE, n_jobs=-1),
}
for name, m in models.items():
    m.fit(X_train, y_train)
    print(name, accuracy_score(y_train, m.predict(X_train)),
          accuracy_score(y_val, m.predict(X_val)))

---
## Задание 2. n_estimators (~15 мин)

График validation accuracy vs `n_estimators` in `[1,5,20,60,150,300]`.

### Подробные критерии (для проверки LLM)

- Выполнены все действия, перечисленные в условии задания.
- Проверочный ориентир: кривая выходит на плато.
- Если задание требует код, код запускается без ошибок и создаёт названные в условии переменные, таблицы или графики.
- Если задание требует текстовый вывод, вывод опирается на полученные числа, таблицы или графики, а не на общие рассуждения.

### Снижение баллов

- Отсутствует ключевой объект из условия (переменная, таблица, график или текстовый вывод) → существенное снижение.
- Использована не та выборка для обучения, подбора или оценки качества → существенное снижение.
- Код не запускается из-за ошибки или результат не связан с условием задания → существенное снижение.
- Текстовый вывод не объясняет полученный результат или противоречит числам/графикам → снижение.


In [ ]:
ns = [1, 5, 20, 60, 150, 300]
val_scores = []
for n in ns:
    m = RandomForestClassifier(n_estimators=n, random_state=RANDOM_STATE, n_jobs=-1)
    m.fit(X_train, y_train)
    val_scores.append(accuracy_score(y_val, m.predict(X_val)))
plt.plot(ns, val_scores, 'o-')
plt.xlabel('n_estimators')
plt.ylabel('validation accuracy')
plt.title('Плато качества леса')
plt.show()

---
## Задание 3. OOB-оценка (~10 мин)

`RandomForestClassifier(oob_score=True, n_estimators=300)`.

OOB (out-of-bag) — оценка качества на тех train-объектах, которые не попали в bootstrap-выборку конкретных деревьев. Сравните OOB-оценку и accuracy на validation.

### Подробные критерии (для проверки LLM)

- Выполнены все действия, перечисленные в условии задания.
- Проверочный ориентир: напечатаны OOB-оценка и validation accuracy.
- Если задание требует код, код запускается без ошибок и создаёт названные в условии переменные, таблицы или графики.
- Если задание требует текстовый вывод, вывод опирается на полученные числа, таблицы или графики, а не на общие рассуждения.

### Снижение баллов

- Отсутствует ключевой объект из условия (переменная, таблица, график или текстовый вывод) → существенное снижение.
- Использована не та выборка для обучения, подбора или оценки качества → существенное снижение.
- Код не запускается из-за ошибки или результат не связан с условием задания → существенное снижение.
- Текстовый вывод не объясняет полученный результат или противоречит числам/графикам → снижение.


In [ ]:
rf = RandomForestClassifier(
    n_estimators=300, oob_score=True, random_state=RANDOM_STATE, n_jobs=-1,
)
rf.fit(X_train, y_train)
print('OOB-оценка:', rf.oob_score_)
print('validation accuracy:', accuracy_score(y_val, rf.predict(X_val)))

---
## Задание 4. max_depth (~12 мин)

Проверьте значения `max_depth` из списка `[None, 5, 10, 20]` при `n_estimators=150`. Выберите лучший вариант по validation accuracy.

Если несколько вариантов дают одинаковое качество, выберите более простую модель: меньшую глубину.

### Подробные критерии (для проверки LLM)

- Выполнены все действия, перечисленные в условии задания.
- Проверочный ориентир: `best_depth` выбран и есть таблица сравнения.
- Если задание требует код, код запускается без ошибок и создаёт названные в условии переменные, таблицы или графики.
- Если задание требует текстовый вывод, вывод опирается на полученные числа, таблицы или графики, а не на общие рассуждения.

### Снижение баллов

- Отсутствует ключевой объект из условия (переменная, таблица, график или текстовый вывод) → существенное снижение.
- Использована не та выборка для обучения, подбора или оценки качества → существенное снижение.
- Код не запускается из-за ошибки или результат не связан с условием задания → существенное снижение.
- Текстовый вывод не объясняет полученный результат или противоречит числам/графикам → снижение.


In [ ]:
best_depth, best_acc = None, -1
for d in [None, 5, 10, 20]:
    m = RandomForestClassifier(
        n_estimators=150, max_depth=d, random_state=RANDOM_STATE, n_jobs=-1,
    )
    m.fit(X_train, y_train)
    acc = accuracy_score(y_val, m.predict(X_val))
    print(d, acc)
    if acc > best_acc:
        best_acc, best_depth = acc, d
print('best_depth:', best_depth)

---
## Задание 5. Bootstrap вручную (~12 мин)

Сделайте 100 повторов bootstrap длины `len(X_train)`.

Bootstrap означает: случайно выбираем индексы объектов **с возвращением**, поэтому один объект может попасть несколько раз, а другой — ни разу.

В каждом повторе посчитайте долю уникальных индексов, затем найдите среднюю долю уникальных.

### Подробные критерии (для проверки LLM)

- Выполнены все действия, перечисленные в условии задания.
- Проверочный ориентир: средняя доля уникальных примерно около 0.63 при большом n.
- Если задание требует код, код запускается без ошибок и создаёт названные в условии переменные, таблицы или графики.
- Если задание требует текстовый вывод, вывод опирается на полученные числа, таблицы или графики, а не на общие рассуждения.

### Снижение баллов

- Отсутствует ключевой объект из условия (переменная, таблица, график или текстовый вывод) → существенное снижение.
- Использована не та выборка для обучения, подбора или оценки качества → существенное снижение.
- Код не запускается из-за ошибки или результат не связан с условием задания → существенное снижение.
- Текстовый вывод не объясняет полученный результат или противоречит числам/графикам → снижение.


In [ ]:
rng = np.random.default_rng(RANDOM_STATE)
n = len(X_train)
fracs = [len(np.unique(rng.choice(n, n, replace=True))) / n for _ in range(100)]
print('средняя доля уникальных:', round(np.mean(fracs), 3))

---
## Задание 6. Permutation importance — важность через перемешивание (~15 мин)

Посчитайте permutation importance на validation для лучшего `RandomForestClassifier` из предыдущих заданий. Идея такая: по очереди перемешиваем один признак и смотрим, насколько падает качество модели.

Выведите top-5 признаков: таблицей или горизонтальным столбчатым графиком (`barh`).

### Подробные критерии (для проверки LLM)

- Выполнены все действия, перечисленные в условии задания.
- Проверочный ориентир: показаны top-5 признаков и понятно, по какой модели они посчитаны.
- Если задание требует код, код запускается без ошибок и создаёт названные в условии переменные, таблицы или графики.
- Если задание требует текстовый вывод, вывод опирается на полученные числа, таблицы или графики, а не на общие рассуждения.

### Снижение баллов

- Отсутствует ключевой объект из условия (переменная, таблица, график или текстовый вывод) → существенное снижение.
- Использована не та выборка для обучения, подбора или оценки качества → существенное снижение.
- Код не запускается из-за ошибки или результат не связан с условием задания → существенное снижение.
- Текстовый вывод не объясняет полученный результат или противоречит числам/графикам → снижение.


In [ ]:
rf = RandomForestClassifier(n_estimators=150, random_state=RANDOM_STATE, n_jobs=-1)
rf.fit(X_train, y_train)
r = permutation_importance(rf, X_val, y_val, n_repeats=10, random_state=RANDOM_STATE)
imp = r.importances_mean
top = np.argsort(imp)[-5:]
for i in top[::-1]:
    print(f'f{i}', round(imp[i], 4))

---
## Задание 7. Разрыв между train и validation (~8 мин)

В markdown объясните: почему высокий train accuracy у случайного леса сам по себе ещё не доказывает переобучение?

Подсказка: смотреть нужно не только на train, но и на validation/OOB-оценку. Если validation accuracy или OOB-оценка тоже хорошие, высокий train accuracy менее тревожен.

### Подробные критерии (для проверки LLM)

- Выполнены все действия, перечисленные в условии задания.
- Проверочный ориентир: 2–3 предложения с опорой на train, validation и/или OOB-оценку.
- Если задание требует код, код запускается без ошибок и создаёт названные в условии переменные, таблицы или графики.
- Если задание требует текстовый вывод, вывод опирается на полученные числа, таблицы или графики, а не на общие рассуждения.

### Снижение баллов

- Отсутствует ключевой объект из условия (переменная, таблица, график или текстовый вывод) → существенное снижение.
- Использована не та выборка для обучения, подбора или оценки качества → существенное снижение.
- Код не запускается из-за ошибки или результат не связан с условием задания → существенное снижение.
- Текстовый вывод не объясняет полученный результат или противоречит числам/графикам → снижение.


# Лес усредняет много деревьев: на train каждое дерево может подгоняться,
# но ансамбль стабильнее одного дерева. Большой разрыв train–val бывает,
# но добавление деревьев реже ухудшает validation — в отличие от одной глубокой модели.

---
## Задание 8. Итог (~5 мин)

Сравните bagging и RF в одном предложении каждый.

### Подробные критерии (для проверки LLM)

- Выполнены все действия, перечисленные в условии задания.
- Проверочный ориентир: markdown.
- Если задание требует код, код запускается без ошибок и создаёт названные в условии переменные, таблицы или графики.
- Если задание требует текстовый вывод, вывод опирается на полученные числа, таблицы или графики, а не на общие рассуждения.

### Снижение баллов

- Отсутствует ключевой объект из условия (переменная, таблица, график или текстовый вывод) → существенное снижение.
- Использована не та выборка для обучения, подбора или оценки качества → существенное снижение.
- Код не запускается из-за ошибки или результат не связан с условием задания → существенное снижение.
- Текстовый вывод не объясняет полученный результат или противоречит числам/графикам → снижение.


# Bagging: много деревьев на разных bootstrap-выборках, усреднение снижает разброс.
# RF: то же + случайный поднабор признаков в узлах → деревья разнообразнее.